# Team 7 · Export the Material Finder v3 (supervised autoencoder)

v3 is the winner of `supervised_ae_final.ipynb`. It is v2 with **one change in training**: the feature autoencoder also sees
the material labels.

**photo → centre crop (middle half) → 224 px → frozen DINOv2-small (768 features) → our feature autoencoder → 32-number code (length 1) → cosine search in a 3,900-patch gallery → ranked materials**

- **v2 loss:** rebuild error only (denoising, noise 0.5).
- **v3 loss:** rebuild error **+ 0.3 × supervised contrastive loss** on the length-1 code: codes of the same material
  are pulled together, codes of different materials pushed apart. λ = 0.3 was chosen on the validation set.

Everything else is the same as v2: the same network (768 → 256 → **32** → 256 → 768), the same features, split,
seed, gallery (same 3,900 patches), preprocessing, ONNX inputs and outputs, and the same answer rule. So the model
server and the app need no code change: only the files on Hugging Face change.

All final numbers below are measured **with the ONNX file**, on the test photos, as the server runs it. The v2 numbers
are read from v2's `config.json` (kept in `models_v2_dino_ae/`).

In [1]:
import io
import json
import shutil
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import polars as pl
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel

AE_DIR = Path("..")
DATA_DIR = AE_DIR / ".." / "data"
MINC_DIR = DATA_DIR / "minc-2500" / "data"
FIG_DIR = AE_DIR / "figures"
MODEL_DIR = AE_DIR / "models"          # goes to Hugging Face
APP_DIR = AE_DIR / "model_app"         # what the Tuesday server needs
for folder in [MODEL_DIR / "gallery_thumbs", MODEL_DIR / "figures", MODEL_DIR / "examples", APP_DIR / "examples"]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ALL_CLASSES = ["brick", "carpet", "ceramic", "fabric", "foliage", "food", "glass", "hair", "leather",
               "metal", "mirror", "other", "painted", "paper", "plastic", "polishedstone", "skin", "sky",
               "stone", "tile", "wallpaper", "water", "wood"]
BUILDING = ["brick", "carpet", "ceramic", "glass", "metal", "mirror", "painted", "plastic",
            "polishedstone", "stone", "tile", "wallpaper", "wood"]
BUILDING_IDS = [ALL_CLASSES.index(c) for c in BUILDING]
OTHER_IDS = [i for i, c in enumerate(ALL_CLASSES) if c not in BUILDING]
remap = {ALL_CLASSES.index(c): k for k, c in enumerate(BUILDING)}
N_CLASSES = len(BUILDING)
print(f"PyTorch {torch.__version__}, ONNX Runtime {ort.__version__}, device {DEVICE}")

/Users/snehashenoy/Downloads/group7_day1_assigment/hackathon-team7-MINC_2500_material_patches/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.14.1, ONNX Runtime 1.31.1, device mps


# 1. Train the supervised feature autoencoder

Cached DINOv2 centre-crop features of the 27,625 training patches, standardised with training statistics.
768 → 256 → **32** → 256 → 768, denoising (noise 0.5) + weight decay 1e-4, 30 epochs, batch 256, seed 42, on the CPU
(the same recipe as `supervised_ae_final.ipynb`, which reported the seed-42 model among its three seeds).

In [2]:
from torch.nn import functional as F

cache = {s: np.load(DATA_DIR / f"minc_round2_{s}.npz") for s in ["train", "val", "test", "test_other"]}
F_train = cache["train"]["dino_centre"]
y_train = np.array([remap[v] for v in cache["train"]["y"]])
y_test = np.array([remap[v] for v in cache["test"]["y"]])
FEAT_MU, FEAT_SD = F_train.mean(0), F_train.std(0) + 1e-6
LAMBDA, TEMPERATURE = 0.3, 0.1


class FeatureAE(nn.Module):
    # Same layers as v2. The forward pass the server uses is encoder + decoder only.
    def __init__(self, n_in=768, latent_dim=32, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_in))

    def forward(self, x):
        return self.decoder(self.encoder(x))


def supcon_loss(z, labels, temperature=TEMPERATURE):
    # Supervised contrastive loss (Khosla et al. 2020) on length-1 codes
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temperature
    eye = torch.eye(len(z), dtype=torch.bool)
    sim = sim.masked_fill(eye, -1e9)
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    positive = (labels[:, None] == labels[None, :]) & ~eye
    return -(log_prob * positive).sum(1).div(positive.sum(1).clamp(min=1)).mean()


Fs_train = ((F_train - FEAT_MU) / FEAT_SD).astype(np.float32)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(4)
ae = FeatureAE()
opt = torch.optim.AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.from_numpy(Fs_train), torch.from_numpy(y_train)), batch_size=256, shuffle=True,
                    generator=torch.Generator().manual_seed(RANDOM_STATE))
start, history = time.time(), []
for epoch in range(30):
    ae.train()
    totals = np.zeros(2)
    for x, yb in loader:
        z = ae.encoder(x + 0.5 * torch.randn_like(x))
        rebuild = F.mse_loss(ae.decoder(z), x)
        label = supcon_loss(z, yb)
        loss = rebuild + LAMBDA * label
        opt.zero_grad(); loss.backward(); opt.step()
        totals += [rebuild.item(), label.item()]
    history.append(totals / len(loader))
ae = ae.eval()
history = np.array(history)
print(f"trained in {time.time() - start:.0f} s, {sum(p.numel() for p in ae.parameters()):,} parameters")
print(f"rebuild error {history[0, 0]:.3f} -> {history[-1, 0]:.3f} | label loss {history[0, 1]:.3f} -> {history[-1, 1]:.3f}")

# The app's gallery: 300 random training patches per material (same seed and rule as v2: same 3,900 patches)
rng = np.random.default_rng(RANDOM_STATE)
gallery_idx = np.concatenate([rng.choice(np.flatnonzero(y_train == k), 300, replace=False) for k in range(N_CLASSES)])
y_gal = y_train[gallery_idx]
old_meta = json.loads((MODEL_DIR / "gallery_meta.json").read_text())
assert [m["id"] for m in old_meta] == gallery_idx.tolist(), "the gallery must be the same patches as v2"
print("gallery:", len(gallery_idx), "(same patches as v2)")

trained in 24 s, 410,912 parameters
rebuild error 0.762 -> 0.440 | label loss 4.431 -> 3.682
gallery: 3900 (same patches as v2)


# 2. One model for the server: `MaterialFinder`

Input: `image`, float32 (batch, 3, 224, 224), pixels 0–1, already cropped and resized. Outputs:
- `code` (batch, 32): the fingerprint, scaled to length 1, so Euclidean search in the gallery = cosine search;
- `unusual_score` (batch,): how badly the autoencoder rebuilds the DINOv2 features (high = unlike building materials).

In [3]:
class MaterialFinder(nn.Module):
    def __init__(self, dino, ae, feat_mu, feat_sd):
        super().__init__()
        self.dino, self.ae = dino, ae
        self.register_buffer("img_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.register_buffer("feat_mu", torch.from_numpy(feat_mu).float())
        self.register_buffer("feat_sd", torch.from_numpy(feat_sd).float())

    def forward(self, image):
        x = (image - self.img_mean) / self.img_std
        h = self.dino(pixel_values=x).last_hidden_state                     # (B, 1 + 256, 384)
        f = torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)                   # class token + mean patch token
        fs = (f - self.feat_mu) / self.feat_sd
        z = self.ae.encoder(fs)
        unusual_score = ((self.ae.decoder(z) - fs) ** 2).mean(1)
        code = z / z.norm(dim=1, keepdim=True)
        return code, unusual_score


dino = AutoModel.from_pretrained("facebook/dinov2-small", attn_implementation="eager").eval()
finder = MaterialFinder(dino, ae, FEAT_MU, FEAT_SD).eval()

ONNX_PATH = MODEL_DIR / "material_finder.onnx"
start = time.time()
torch.onnx.export(finder, (torch.rand(2, 3, 224, 224),), str(ONNX_PATH), dynamo=False, opset_version=17,
                  input_names=["image"], output_names=["code", "unusual_score"],
                  dynamic_axes={"image": {0: "batch"}, "code": {0: "batch"}, "unusual_score": {0: "batch"}})
print(f"exported in {time.time() - start:.0f} s: {ONNX_PATH.name}, {ONNX_PATH.stat().st_size / 1e6:.1f} MB")
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

W1118 13:19:19.864000 18248 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7154.07it/s]


/var/folders/7x/4mr6dq0s3ll7hrz5x39p97680000gp/T/ipykernel_18248/319757337.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(finder, (torch.rand(2, 3, 224, 224),), str(ONNX_PATH), dynamo=False, opset_version=17,
/Users/snehashenoy/Downloads/group7_day1_assigment/hackathon-team7-MINC_2500_material_patches/.venv/lib/python3.12/site-packages/transformers/models/dinov2/modeling_dinov2.py:62: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to 

exported in 1 s: material_finder.onnx, 90.1 MB


# 3. Preprocessing (the server must do exactly this) and the ONNX check

In [4]:
def preprocess(image: Image.Image) -> np.ndarray:
    # RGB -> centre square -> its middle half -> 224 x 224 (bilinear) -> 0-1 -> (1, 3, 224, 224) float32
    image = image.convert("RGB")
    w, h = image.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    image = image.crop((left, top, left + side, top + side))
    half = int(side * 0.5)
    off = (side - half) // 2
    image = image.crop((off, off, off + half, off + half)).resize((224, 224), Image.BILINEAR)
    return (np.asarray(image, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]


def load_bytes(split, label_ids):
    files = sorted(MINC_DIR.glob(f"{split}-*.parquet"))
    df = (pl.scan_parquet(files).select(pl.col("image").struct.field("bytes").alias("bytes"), pl.col("label"))
          .filter(pl.col("label").is_in(label_ids)).collect())
    return df["bytes"].to_list(), df["label"].to_numpy()


def run_onnx(byte_list, batch_size=32):
    codes, scores = [], []
    for i in range(0, len(byte_list), batch_size):
        x = np.concatenate([preprocess(Image.open(io.BytesIO(b))) for b in byte_list[i:i + batch_size]])
        c, s = session.run(None, {"image": x})
        codes.append(c); scores.append(s)
    return np.concatenate(codes), np.concatenate(scores)


raw = {}
for split, ids, name in [("train", BUILDING_IDS, "train"), ("validation", BUILDING_IDS, "val"),
                         ("test", BUILDING_IDS, "test"), ("test", OTHER_IDS, "test_other")]:
    raw[name], labels = load_bytes(split, ids)
    assert (labels == cache[name]["y"]).all(), f"order of {name} differs from the cache"
print({k: len(v) for k, v in raw.items()})

# ONNX vs PyTorch on the same 16 test photos
x = np.concatenate([preprocess(Image.open(io.BytesIO(b))) for b in raw["test"][:16]])
with torch.no_grad():
    c_pt, s_pt = finder(torch.from_numpy(x))
c_ox, s_ox = session.run(None, {"image": x})
print(f"code: max difference {np.abs(c_ox - c_pt.numpy()).max():.2e} | "
      f"unusual score: max relative difference {np.abs(s_ox / s_pt.numpy() - 1).max():.2e}")
assert np.allclose(c_ox, c_pt.numpy(), atol=1e-3)

{'train': 27625, 'val': 1625, 'test': 3250, 'test_other': 2500}


code: max difference 7.45e-07 | unusual score: max relative difference 3.81e-06


# 4. Gallery, threshold and final numbers (all with the ONNX file)

About 11,000 photos through DINOv2 on the CPU: a few minutes.

In [5]:
start = time.time()
Z_gal, _ = run_onnx([raw["train"][i] for i in gallery_idx])
_, s_val = run_onnx(raw["val"])
Z_test, s_test = run_onnx(raw["test"])
_, s_other = run_onnx(raw["test_other"])
print(f"ONNX on {len(gallery_idx) + len(raw['val']) + len(raw['test']) + len(raw['test_other']):,} photos "
      f"in {time.time() - start:.0f} s")

THRESHOLD = float(np.percentile(s_val, 95))
K, MIN_SHARE = 5, 0.2
dist, nbr = NearestNeighbors(n_neighbors=K).fit(Z_gal).kneighbors(Z_test)
votes = y_gal[nbr]
shares = np.stack([(votes == k).mean(1) for k in range(N_CLASSES)], axis=1)
closest = np.where(votes[:, :, None] == np.arange(N_CLASSES), dist[:, :, None], np.inf).min(1)
order = np.argsort(-(shares - 1e-6 * np.where(np.isinf(closest), 0, closest)), axis=1, kind="stable")
top = lambda k: ((order[:, :k] == y_test[:, None]) & (np.take_along_axis(shares, order[:, :k], 1) > 0)).any(1).mean()
shown = np.take_along_axis(shares, order[:, :3], 1) >= MIN_SHARE
shown[:, 0] = True

results = {
    "precision_at_5_gallery": float((votes == y_test[:, None]).mean()),
    "top1_accuracy": float(top(1)), "top2_accuracy": float(top(2)), "top3_accuracy": float(top(3)),
    "right_material_shown_min_share_0_2": float(((order[:, :3] == y_test[:, None]) & shown).any(1).mean()),
    "suggestions_shown_avg": float(shown.sum(1).mean()),
    "unusual_roc_auc": float(roc_auc_score(np.r_[np.zeros(len(s_test)), np.ones(len(s_other))], np.r_[s_test, s_other])),
    "unusual_flagged_building": float((s_test > THRESHOLD).mean()),
    "unusual_flagged_other": float((s_other > THRESHOLD).mean()),
}
v2_results = json.loads((AE_DIR / "models_v2_dino_ae" / "config.json").read_text())["results"]
pd.DataFrame({"v2": pd.Series(v2_results), "v3": pd.Series(results)}).drop(index="precision_at_5_full_train", errors="ignore").round(3)

ONNX on 11,275 photos in 421 s


,v2,v3
precision_at_5_gallery,0.607,0.768
right_material_shown_min_share_0_2,0.846,0.879
suggestions_shown_avg,2.006,1.371
top1_accuracy,0.698,0.798
top2_accuracy,0.812,0.867
top3_accuracy,0.846,0.879
unusual_flagged_building,0.051,0.048
unusual_flagged_other,0.413,0.428
unusual_roc_auc,0.818,0.829


In [6]:
# Unusual score per other class: which non-building photos get flagged?
other_names = [ALL_CLASSES[v] for v in cache["test_other"]["y"]]
(pd.DataFrame({"class": other_names, "flagged": s_other > THRESHOLD})
 .groupby("class")["flagged"].mean().sort_values().to_frame("share flagged").T)

class,paper,leather,fabric,food,other,water,foliage,sky,hair,skin
share flagged,0.124,0.248,0.26,0.264,0.336,0.46,0.464,0.608,0.748,0.768


# 5. Write the files for Hugging Face (`models/`) and the server (`model_app/`)

In [7]:
# Same gallery patches and thumbnails as v2 (the 256 px thumbnails from scripts/gallery_thumbs_256.py stay as they are):
# only the codes change
meta = old_meta
np.save(MODEL_DIR / "gallery_latents.npy", Z_gal.astype(np.float32))
(MODEL_DIR / "gallery_meta.json").write_text(json.dumps(meta))

torch.save({"feature_ae": ae.state_dict(), "feat_mu": FEAT_MU, "feat_sd": FEAT_SD, "lambda": LAMBDA,
            "backbone": "facebook/dinov2-small"}, MODEL_DIR / "material_finder_ae.pt")

config = {
    "model": "material_finder.onnx",
    "version": 3,
    "setup": "frozen DINOv2-small (centre crop, 224 px) + supervised feature autoencoder 768-256-32 (rebuild + 0.3 x supervised contrastive loss on the code), cosine distance",
    "training": {"loss": "MSE rebuild (denoising, noise 0.5) + 0.3 x supervised contrastive (temperature 0.1) on the length-1 code", "epochs": 30, "batch_size": 256, "weight_decay": 1e-4, "seed": RANDOM_STATE, "chosen_in": "supervised_ae_final.ipynb"},
    "backbone": "facebook/dinov2-small (frozen, not trained by us)",
    "latent_dim": 32,
    "image_size": 224,
    "preprocessing": "RGB, centre square, keep its middle half, resize to 224x224 (PIL bilinear), pixels / 255 -> 0-1, "
                     "shape (1, 3, 224, 224) float32. ImageNet normalisation happens inside the ONNX file.",
    "inputs": {"image": "float32 (batch, 3, 224, 224), values 0-1"},
    "outputs": {"code": "float32 (batch, 32), length 1", "unusual_score": "float32 (batch,)"},
    "classes": BUILDING,
    "similarity": "Euclidean distance between length-1 codes (= cosine); 5 nearest gallery patches",
    "neighbours": K,
    "answer": "ranked materials by share of the 5 neighbours; show materials with share >= min_share, at most 3",
    "min_share": MIN_SHARE,
    "gallery_size": len(gallery_idx),
    "unusual_threshold": THRESHOLD,
    "unusual_threshold_rule": "95th percentile of unusual_score on validation building patches",
    "trained_parameters": sum(p.numel() for p in ae.parameters()),
    "results": results,
    "dataset": "MINC-2500 (mcimpoi/minc-2500_split_1), CC BY 4.0, Bell et al. 2015 CVPR",
}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))

for name in ["01_samples", "09_improvements_precision", "12_round2_precision", "13_round2_per_material",
             "15_top_materials_accuracy", "16_top_materials_calibration", "17_top_materials_examples",
             "18_centre_crop_similarity", "24_supervised_ae_results", "25_supervised_ae_per_material"]:
    src = FIG_DIR / f"{name}.png"
    src = src if src.exists() else AE_DIR / "notebook" / f"{name}.png"
    shutil.copy(src, MODEL_DIR / "figures" / f"{name}.png")

# Example photos: one test patch per material for 4 materials (original 362 px JPEGs)
for m in ["brick", "carpet", "tile", "wood"]:
    i = int(np.flatnonzero(y_test == BUILDING.index(m))[0])
    (MODEL_DIR / "examples" / f"{m}.jpg").write_bytes(raw["test"][i])

for f in ["material_finder.onnx", "config.json"]:
    shutil.copy(MODEL_DIR / f, APP_DIR / f)
for f in (MODEL_DIR / "examples").glob("*.jpg"):
    shutil.copy(f, APP_DIR / "examples" / f.name)
for p in sorted(MODEL_DIR.rglob("*")):
    if p.is_file() and "gallery_thumbs" not in p.parts:
        print(f"{p.relative_to(MODEL_DIR)}: {p.stat().st_size / 1e6:.2f} MB")
print(f"gallery_thumbs/: {len(list((MODEL_DIR / 'gallery_thumbs').glob('*.jpg')))} files")

README.md: 0.01 MB
config.json: 0.00 MB
examples/brick.jpg: 0.03 MB
examples/carpet.jpg: 0.03 MB
examples/tile.jpg: 0.03 MB
examples/wood.jpg: 0.04 MB
figures/01_samples.png: 0.49 MB
figures/09_improvements_precision.png: 0.13 MB
figures/12_round2_precision.png: 0.08 MB
figures/13_round2_per_material.png: 0.05 MB
figures/15_top_materials_accuracy.png: 0.06 MB
figures/16_top_materials_calibration.png: 0.05 MB
figures/17_top_materials_examples.png: 1.29 MB
figures/18_centre_crop_similarity.png: 1.11 MB
figures/19_cc0_swatches.png: 1.58 MB
figures/24_supervised_ae_results.png: 0.08 MB
figures/25_supervised_ae_per_material.png: 0.11 MB
gallery_clusters.json: 0.34 MB
gallery_latents.npy: 0.50 MB
gallery_meta.json: 0.29 MB
material_finder.onnx: 90.05 MB
material_finder_ae.pt: 1.66 MB
swatch_latents.npy: 0.16 MB
swatch_meta.json: 0.45 MB
swatch_spheres/00000.jpg: 0.01 MB
swatch_spheres/00001.jpg: 0.05 MB
swatch_spheres/00002.jpg: 0.03 MB
swatch_spheres/00003.jpg: 0.04 MB
swatch_spheres/00004.

# 6. Final check: the model card's "How to use" code on the example photos

In [8]:
gallery = np.load(MODEL_DIR / "gallery_latents.npy")
meta = json.loads((MODEL_DIR / "gallery_meta.json").read_text())
cfg = json.loads((MODEL_DIR / "config.json").read_text())

for photo in sorted((MODEL_DIR / "examples").glob("*.jpg")):
    code_, score = session.run(None, {"image": preprocess(Image.open(photo))})
    distances = np.linalg.norm(gallery - code_, axis=1)
    top5 = np.argsort(distances)[:5]
    materials = [meta[i]["material"] for i in top5]
    ranked = sorted(set(materials), key=lambda m: (-materials.count(m), materials.index(m)))
    answer = " · ".join(f"{m} {materials.count(m) / 5:.0%}" for m in ranked[:3] if materials.count(m) / 5 >= cfg["min_share"])
    print(f"{photo.name:11s} -> {answer:40s} unusual: {bool(score[0] > cfg['unusual_threshold'])}")

brick.jpg   -> brick 100%                               unusual: False
carpet.jpg  -> wood 60% · metal 20% · carpet 20%        unusual: False
tile.jpg    -> tile 100%                                unusual: False
wood.jpg    -> painted 100%                             unusual: False


# 7. The clean swatches: new codes with v3

The 1,280 CC0 swatches keep their pictures, names and colours (`swatch_meta.json`); only their codes are recomputed with
the v3 ONNX file, from the same 512 px sphere previews and the same preprocessing as `cc0_swatches_final.ipynb`.
Then the same sanity check: for the MINC test photos of materials that exist as swatches, is the nearest swatch
(or any of the 5 nearest) of a matching material? v2 found: nearest 40 %, any of 5: 65 %.

In [9]:

from collections import Counter

RAW_DIR = DATA_DIR / "swatches" / "raw"
BACKGROUND = (18, 18, 18)
sw_meta = json.loads((MODEL_DIR / "swatch_meta.json").read_text())


def on_background(path):
    image = Image.open(path).convert("RGBA").resize((512, 512), Image.LANCZOS)
    canvas = Image.new("RGBA", image.size, BACKGROUND + (255,))
    return Image.alpha_composite(canvas, image).convert("RGB")


start = time.time()
Z_swatch = np.concatenate([session.run(None, {"image": np.concatenate(
    [preprocess(on_background(RAW_DIR / f"{m['key']}.img")) for m in sw_meta[i:i + 32]])})[0]
    for i in range(0, len(sw_meta), 32)]).astype(np.float32)
print(f"{len(Z_swatch):,} swatches encoded in {time.time() - start:.0f} s")

MATCH = {"brick": {"brick"}, "wood": {"wood"}, "metal": {"metal"}, "stone": {"stone", "rock", "paving"},
         "polishedstone": {"marble", "stone"}, "tile": {"tile", "ceramic"}, "ceramic": {"ceramic", "tile"},
         "plastic": {"plastic"}, "painted": {"plaster", "concrete"}}
sw_mat = np.array([m["material"] for m in sw_meta])


def swatch_check(Z_sw, Z_photos):
    keep = np.isin(y_test, [BUILDING.index(c) for c in MATCH])
    labels = [BUILDING[v] for v in y_test[keep]]
    nearest = NearestNeighbors(n_neighbors=5).fit(Z_sw).kneighbors(Z_photos[keep], return_distance=False)
    hit1 = [sw_mat[n[0]] in MATCH[l] for n, l in zip(nearest, labels)]
    hit5 = [len(set(sw_mat[n]) & MATCH[l]) > 0 for n, l in zip(nearest, labels)]
    t = pd.DataFrame({"MINC class": labels, "nearest": hit1, "any of 5": hit5}).groupby("MINC class").mean()
    t.loc["all"] = [np.mean(hit1), np.mean(hit5)]
    return t


v3_check = swatch_check(Z_swatch, Z_test)
v3_check.columns = [f"v3 {c}" for c in v3_check.columns]
v3_check.round(3)


1,280 swatches encoded in 58 s


,v3 nearest,v3 any of 5
MINC class,,
brick,0.616,0.924
ceramic,0.076,0.144
metal,0.688,0.864
painted,0.220,0.816
plastic,0.088,0.336
polishedstone,0.732,0.824
stone,0.580,0.876
tile,0.596,0.880
wood,0.804,0.884


In [10]:

np.save(MODEL_DIR / "swatch_latents.npy", Z_swatch)
config = json.loads((MODEL_DIR / "config.json").read_text())
config["results"]["swatch_nearest_matches"] = float(v3_check.loc["all", "v3 nearest"])
config["results"]["swatch_any_of_5_matches"] = float(v3_check.loc["all", "v3 any of 5"])
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))
shutil.copy(MODEL_DIR / "config.json", APP_DIR / "config.json")
print("swatch_latents.npy and config.json written:", config["results"]["swatch_nearest_matches"],
      config["results"]["swatch_any_of_5_matches"])


swatch_latents.npy and config.json written: 0.4888888888888889 0.7275555555555555
